# 第一章：项目概览与架构 — 交互式笔记本

> 在这个 notebook 里，你将动手构建 mini-claude 的类型基础，并通过交互式代码理解 Claude Code 的核心数据结构。
>
> **运行环境**: [Deno Jupyter Kernel](https://docs.deno.com/runtime/reference/cli/jupyter/) — 原生 TypeScript 支持
>
> **配套文档**: [docs/zh-CN/01-overview.md](../docs/zh-CN/01-overview.md)

---
## 1. 理解 Content Block：AI 回复的最小单元

Claude API 的回复不是纯文本，而是由多种 **Content Block** 组成的数组。

就像 HTML 页面由 `<p>`、`<img>`、`<div>` 等元素组成一样，AI 的一条回复由以下几种 block 组成：

In [ ]:
// --- Content Block 类型定义 ---

// AI 生成的文字
interface TextBlock {
  type: "text";
  text: string;
}

// AI 决定调用某个工具
interface ToolUseBlock {
  type: "tool_use";
  id: string;       // 唯一标识，用于匹配后续的 tool_result
  name: string;     // 工具名，如 "Bash"、"Read"、"Grep"
  input: Record<string, unknown>;  // 工具参数
}

// 工具执行后返回的结果
interface ToolResultBlock {
  type: "tool_result";
  tool_use_id: string;  // 对应 ToolUseBlock.id
  content: string;
  is_error?: boolean;
}

// Discriminated Union — 通过 type 字段区分
type ContentBlock = TextBlock | ToolUseBlock | ToolResultBlock;

console.log("✅ Content Block 类型定义完成");

### 动手试试：构造一条 AI 回复

当用户说 "帮我看看 main.ts"，AI 的回复通常是这样的：

In [ ]:
// 模拟 AI 的一次回复：先说一句话，然后调用工具
const aiResponse: ContentBlock[] = [
  { type: "text", text: "好的，让我读取这个文件。" },
  {
    type: "tool_use",
    id: "tool_001",
    name: "Read",
    input: { file_path: "main.ts" },
  },
];

// 遍历内容块，展示 discriminated union 的类型收窄
for (const block of aiResponse) {
  switch (block.type) {
    case "text":
      console.log(`📝 文本: "${block.text}"`);
      break;
    case "tool_use":
      console.log(`🔧 工具调用: ${block.name}(${JSON.stringify(block.input)})`);
      break;
    case "tool_result":
      console.log(`📋 工具结果: ${block.content}`);
      break;
  }
}

> **练习**: 修改上面的 `aiResponse` 数组，添加一个 `tool_result` 块，模拟文件读取的结果。

---
## 2. 消息类型：对话历史的结构

Content Block 是消息的"原子"，Message 是对话历史的"单元"。

In [ ]:
// --- Message 类型定义 ---

type StopReason = "end_turn" | "tool_use" | "max_tokens";

interface UserMessage {
  type: "user";
  uuid: string;
  message: {
    role: "user";
    content: string | ContentBlock[];
  };
}

interface AssistantMessage {
  type: "assistant";
  uuid: string;
  message: {
    role: "assistant";
    content: ContentBlock[];
    model: string;
    stop_reason: StopReason | null;
  };
}

interface SystemMessage {
  type: "system";
  subtype: "info" | "compact_boundary" | "local_command";
  message: string;
}

type Message = UserMessage | AssistantMessage | SystemMessage;

console.log("✅ Message 类型定义完成");

### 动手试试：模拟一次完整的工具调用循环

Agentic Loop 的核心就是一个不断增长的消息数组。下面我们模拟一次完整的循环：

In [ ]:
// 模拟一次完整的 Agentic Loop 消息流
const conversationHistory: Message[] = [];

// Step 1: 用户提问
conversationHistory.push({
  type: "user",
  uuid: crypto.randomUUID(),
  message: {
    role: "user",
    content: "项目根目录下有哪些 TypeScript 文件？",
  },
});

// Step 2: AI 决定调用 Glob 工具
conversationHistory.push({
  type: "assistant",
  uuid: crypto.randomUUID(),
  message: {
    role: "assistant",
    content: [
      { type: "text", text: "我来帮你查看一下。" },
      {
        type: "tool_use",
        id: "call_001",
        name: "Glob",
        input: { pattern: "*.ts" },
      },
    ],
    model: "claude-sonnet-4-20250514",
    stop_reason: "tool_use",  // ← 这告诉 QueryEngine: 别停，还有活要干
  },
});

// Step 3: 工具执行结果作为 user 消息发回
conversationHistory.push({
  type: "user",
  uuid: crypto.randomUUID(),
  message: {
    role: "user",
    content: [
      {
        type: "tool_result",
        tool_use_id: "call_001",
        content: "main.ts\ntsconfig.json 不匹配\n找到 1 个文件: main.ts",
      },
    ],
  },
});

// Step 4: AI 基于工具结果给出最终回答
conversationHistory.push({
  type: "assistant",
  uuid: crypto.randomUUID(),
  message: {
    role: "assistant",
    content: [
      { type: "text", text: "根目录下有 1 个 TypeScript 文件：main.ts" },
    ],
    model: "claude-sonnet-4-20250514",
    stop_reason: "end_turn",  // ← 这告诉 QueryEngine: 我说完了，循环结束
  },
});

// 打印完整对话历史
console.log("\n📜 对话历史 (%d 条消息):\n", conversationHistory.length);
for (const msg of conversationHistory) {
  if (msg.type === "user") {
    const content = msg.message.content;
    if (typeof content === "string") {
      console.log(`👤 用户: ${content}`);
    } else {
      for (const block of content) {
        if (block.type === "tool_result") {
          console.log(`📋 [工具结果 → ${block.tool_use_id}]: ${block.content.substring(0, 60)}...`);
        }
      }
    }
  } else if (msg.type === "assistant") {
    for (const block of msg.message.content) {
      if (block.type === "text") console.log(`🤖 助手: ${block.text}`);
      if (block.type === "tool_use") console.log(`🔧 调用: ${block.name}(${JSON.stringify(block.input)})`);
    }
    console.log(`   [stop_reason: ${msg.message.stop_reason}]`);
  }
}

### 关键洞察：stop_reason 控制循环

注意上面的 `stop_reason` 字段：

| stop_reason | 含义 | QueryEngine 行为 |
|-------------|------|------------------|
| `"tool_use"` | AI 需要工具结果 | **继续循环** → 执行工具 → 发回结果 |
| `"end_turn"` | AI 认为任务完成 | **退出循环** → 显示最终回答 |
| `"max_tokens"` | 输出达到上限 | **可能继续** → 请求 AI 继续输出 |

这就是 Agentic Loop 的控制核心——不是一个复杂的状态机，只是一个看 `stop_reason` 的 while 循环。

---
## 3. 工具接口：能力的抽象

Claude Code 的每一项能力（读文件、运行命令、搜索代码）都被抽象为一个 Tool 对象。

In [ ]:
// --- Tool 接口定义 ---

interface ToolResult {
  content: string;
  isError?: boolean;
}

interface JSONSchemaProperty {
  type: "string" | "number" | "boolean" | "array" | "object";
  description?: string;
  enum?: string[];
}

interface JSONSchema {
  type: "object";
  properties: Record<string, JSONSchemaProperty>;
  required?: string[];
}

interface Tool {
  name: string;
  description: string;
  inputSchema: JSONSchema;
  call(input: Record<string, unknown>): Promise<ToolResult>;
  isReadOnly?: boolean;
}

console.log("✅ Tool 接口定义完成");

### 动手试试：创建你的第一个 Tool

让我们实现一个真正能运行的工具——用 Deno API 读取文件：

In [ ]:
// 实现一个真正的 FileRead 工具
const FileReadTool: Tool = {
  name: "Read",
  description: "读取指定文件的内容",
  inputSchema: {
    type: "object",
    properties: {
      file_path: { type: "string", description: "文件的路径" },
    },
    required: ["file_path"],
  },
  isReadOnly: true,
  async call(input) {
    try {
      const path = input.file_path as string;
      const content = await Deno.readTextFile(path);
      return { content };
    } catch (e) {
      return { content: `Error: ${e}`, isError: true };
    }
  },
};

// 实现一个 Bash 工具
const BashTool: Tool = {
  name: "Bash",
  description: "执行 shell 命令",
  inputSchema: {
    type: "object",
    properties: {
      command: { type: "string", description: "要执行的命令" },
    },
    required: ["command"],
  },
  isReadOnly: false,  // shell 命令可能有副作用
  async call(input) {
    try {
      const cmd = new Deno.Command("sh", {
        args: ["-c", input.command as string],
        stdout: "piped",
        stderr: "piped",
      });
      const output = await cmd.output();
      const stdout = new TextDecoder().decode(output.stdout);
      const stderr = new TextDecoder().decode(output.stderr);
      return {
        content: stdout + (stderr ? `\nSTDERR: ${stderr}` : ""),
        isError: !output.success,
      };
    } catch (e) {
      return { content: `Error: ${e}`, isError: true };
    }
  },
};

console.log("✅ FileReadTool 和 BashTool 创建完成");
console.log(`   Read: ${FileReadTool.description} (只读: ${FileReadTool.isReadOnly})`);
console.log(`   Bash: ${BashTool.description} (只读: ${BashTool.isReadOnly})`);

In [ ]:
// 实际运行工具！
console.log("\n--- 运行 BashTool: ls *.ts ---\n");
const lsResult = await BashTool.call({ command: "ls *.ts 2>/dev/null || echo '当前目录无 .ts 文件'" });
console.log(lsResult.content);

console.log("\n--- 运行 BashTool: pwd ---\n");
const pwdResult = await BashTool.call({ command: "pwd" });
console.log(pwdResult.content);

> **练习**: 用 `BashTool` 运行 `echo hello world`，或用 `FileReadTool` 读取一个你知道存在的文件。

---
## 4. 工具注册表与 API 格式

Claude Code 维护一个工具注册表（Tool Registry），在调用 API 时将工具转换为 Anthropic API 需要的格式。

In [ ]:
// 工具注册表
const toolRegistry: Tool[] = [FileReadTool, BashTool];

// 转换为 API 格式
interface APIToolDefinition {
  name: string;
  description: string;
  input_schema: JSONSchema;
}

function toolToAPIFormat(tool: Tool): APIToolDefinition {
  return {
    name: tool.name,
    description: tool.description,
    input_schema: tool.inputSchema,
  };
}

// 查看发给 API 的 tools 参数
const apiTools = toolRegistry.map(toolToAPIFormat);
console.log("发给 Anthropic API 的 tools 参数:\n");
console.log(JSON.stringify(apiTools, null, 2));

### 根据名称查找工具

当 AI 返回 `tool_use` 块时，QueryEngine 需要根据 `name` 找到对应的 Tool 实现并执行：

In [ ]:
// 根据名称查找工具（真实 Claude Code 中的 findToolByName）
function findTool(name: string): Tool | undefined {
  return toolRegistry.find((t) => t.name === name);
}

// 模拟处理一个 tool_use 块
async function executeToolCall(block: ToolUseBlock): Promise<ToolResultBlock> {
  const tool = findTool(block.name);
  if (!tool) {
    return {
      type: "tool_result",
      tool_use_id: block.id,
      content: `Error: Unknown tool '${block.name}'`,
      is_error: true,
    };
  }

  const result = await tool.call(block.input);
  return {
    type: "tool_result",
    tool_use_id: block.id,
    content: result.content,
    is_error: result.isError,
  };
}

// 测试：执行一个工具调用
const toolCall: ToolUseBlock = {
  type: "tool_use",
  id: "call_test_001",
  name: "Bash",
  input: { command: "echo '🎉 工具执行成功！'" },
};

const result = await executeToolCall(toolCall);
console.log("工具调用结果:");
console.log(result);

---
## 5. 权限类型：安全的门控

Claude Code 不会让 AI 随意执行任何操作。每个工具调用前都要经过权限检查。

In [ ]:
// --- 权限类型定义 ---

type PermissionMode = "default" | "auto" | "bypassPermissions";
type PermissionBehavior = "allow" | "deny" | "ask";

interface PermissionRule {
  toolName: string;       // "*" 匹配所有工具
  pattern?: string;       // 可选的命令模式匹配
  behavior: PermissionBehavior;
  reason?: string;
}

// 定义一组权限规则
const permissionRules: PermissionRule[] = [
  // 读操作：始终允许
  { toolName: "Read", behavior: "allow", reason: "只读操作无风险" },
  // 危险命令：始终拒绝
  { toolName: "Bash", pattern: "rm -rf", behavior: "deny", reason: "危险的删除操作" },
  { toolName: "Bash", pattern: ":(){ :|:& };:", behavior: "deny", reason: "Fork 炸弹" },
  // 其他 Bash 命令：需要询问用户
  { toolName: "Bash", behavior: "ask", reason: "Shell 命令可能有副作用" },
];

console.log("✅ 权限规则定义完成:");
for (const rule of permissionRules) {
  const target = rule.pattern ? `${rule.toolName}(${rule.pattern})` : rule.toolName;
  console.log(`  ${rule.behavior.toUpperCase().padEnd(5)} ${target} — ${rule.reason}`);
}

In [ ]:
// 实现一个简单的权限检查器
function checkPermission(
  toolName: string,
  input: Record<string, unknown>,
  rules: PermissionRule[]
): { behavior: PermissionBehavior; reason?: string } {
  for (const rule of rules) {
    // 工具名匹配
    if (rule.toolName !== "*" && rule.toolName !== toolName) continue;
    // 模式匹配（如果有）
    if (rule.pattern) {
      const command = String(input.command ?? "");
      if (!command.includes(rule.pattern)) continue;
    }
    return { behavior: rule.behavior, reason: rule.reason };
  }
  return { behavior: "ask", reason: "默认需要确认" };
}

// 测试各种场景
const testCases = [
  { tool: "Read", input: { file_path: "main.ts" } },
  { tool: "Bash", input: { command: "ls -la" } },
  { tool: "Bash", input: { command: "rm -rf /" } },
  { tool: "Bash", input: { command: "git status" } },
];

console.log("\n权限检查测试:\n");
for (const tc of testCases) {
  const decision = checkPermission(tc.tool, tc.input, permissionRules);
  const icon = decision.behavior === "allow" ? "✅" : decision.behavior === "deny" ? "🚫" : "❓";
  const cmd = tc.input.command ?? tc.input.file_path;
  console.log(`${icon} ${tc.tool}("${cmd}") → ${decision.behavior} (${decision.reason})`);
}

---
## 6. 把它们组合起来：迷你 Agentic Loop

现在我们有了所有的积木。让我们把它们拼成一个最简单的 Agentic Loop 模拟器：

In [ ]:
/**
 * 迷你 Agentic Loop 模拟器
 * 
 * 真正的 QueryEngine 调用 Anthropic API，这里我们用硬编码的
 * AI 回复来模拟，但数据流和消息结构完全一致。
 */

// 模拟 API 返回（真实场景中这来自 Anthropic API 流式响应）
function simulateAPIResponse(turn: number): AssistantMessage {
  if (turn === 0) {
    return {
      type: "assistant",
      uuid: crypto.randomUUID(),
      message: {
        role: "assistant",
        content: [
          { type: "text", text: "让我先看看当前目录的内容。" },
          { type: "tool_use", id: "t1", name: "Bash", input: { command: "ls" } },
        ],
        model: "claude-sonnet-4-20250514",
        stop_reason: "tool_use",
      },
    };
  } else {
    return {
      type: "assistant",
      uuid: crypto.randomUUID(),
      message: {
        role: "assistant",
        content: [
          { type: "text", text: "当前目录的文件列表已获取。任务完成！" },
        ],
        model: "claude-sonnet-4-20250514",
        stop_reason: "end_turn",
      },
    };
  }
}

// Agentic Loop 主函数
async function agenticLoop(userInput: string) {
  const messages: Message[] = [];
  let turn = 0;
  const MAX_TURNS = 10;

  // 添加用户消息
  messages.push({
    type: "user",
    uuid: crypto.randomUUID(),
    message: { role: "user", content: userInput },
  });
  console.log(`👤 用户: ${userInput}\n`);

  // 循环直到 end_turn 或达到最大轮次
  while (turn < MAX_TURNS) {
    console.log(`--- 第 ${turn + 1} 轮 ---`);

    // 1. 调用 API（模拟）
    const response = simulateAPIResponse(turn);
    messages.push(response);

    // 2. 处理回复中的内容块
    for (const block of response.message.content) {
      if (block.type === "text") {
        console.log(`🤖 ${block.text}`);
      }
    }

    // 3. 检查是否需要执行工具
    if (response.message.stop_reason === "end_turn") {
      console.log("\n✅ 循环结束 (stop_reason: end_turn)");
      break;
    }

    // 4. 找到所有工具调用并执行
    const toolUses = response.message.content.filter(
      (b): b is ToolUseBlock => b.type === "tool_use"
    );

    const toolResults: ToolResultBlock[] = [];

    for (const toolUse of toolUses) {
      // 权限检查
      const perm = checkPermission(toolUse.name, toolUse.input, permissionRules);
      if (perm.behavior === "deny") {
        console.log(`🚫 权限拒绝: ${toolUse.name} — ${perm.reason}`);
        toolResults.push({
          type: "tool_result",
          tool_use_id: toolUse.id,
          content: `Permission denied: ${perm.reason}`,
          is_error: true,
        });
        continue;
      }

      // 执行工具
      console.log(`🔧 执行: ${toolUse.name}(${JSON.stringify(toolUse.input)})`);
      const result = await executeToolCall(toolUse);
      console.log(`📋 结果: ${result.content.substring(0, 100)}`);
      toolResults.push(result);
    }

    // 5. 将工具结果作为 user 消息追加到历史
    messages.push({
      type: "user",
      uuid: crypto.randomUUID(),
      message: {
        role: "user",
        content: toolResults,
      },
    });

    turn++;
    console.log();
  }

  console.log(`\n📊 总计: ${messages.length} 条消息, ${turn + 1} 轮对话`);
}

// 运行！
await agenticLoop("看看当前目录有什么文件");

---
## 7. 本章总结

在这个 notebook 中，你：

1. **定义了 Content Block 类型** — AI 回复的最小组成单元
2. **定义了 Message 类型** — 对话历史的基本单元
3. **实现了 Tool 接口** — 并创建了真正能运行的 FileRead 和 Bash 工具
4. **构建了工具注册表** — 理解了工具如何转换为 API 格式
5. **实现了权限检查** — 理解了安全门控机制
6. **组合了迷你 Agentic Loop** — 理解了 Claude Code 最核心的运行循环

### 下一章预告

第二章我们将在 `demo/` 目录中实现真正的 Tool 系统和工具注册表，让 mini-claude 拥有真实的"动手能力"。

### 进一步阅读

- [Anthropic Messages API 文档](https://docs.anthropic.com/en/api/messages)
- [Tool Use 指南](https://docs.anthropic.com/en/docs/build-with-claude/tool-use)
- 源码：`anthhub-claude-code/src/Tool.ts`、`anthhub-claude-code/src/types/message.ts`